In [17]:
import os
os.getcwd()

'G:\\My Drive\\bsd\\11_radiomics_for_gliomas\\02_train\\trial_100\\scripts'

In [18]:
os.listdir()

['01_import_data.ipynb',
 '02_preprocess_images.ipynb',
 '04_normalize_images.py',
 '05_extract_wt_radiomics.py',
 '06_combine_features.py',
 '.ipynb_checkpoints',
 'all_radiomics_features.csv',
 'UTSW_Glioma_Metadata-2-1.tsv',
 'EGD_Clinical_data.xlsx',
 'UCSF-PDGM-metadata_v5 (1).csv',
 '07_create_2021_glioma_labels.ipynb',
 'combined_glioma_who2021.csv',
 'doiJNLP-JAMS4RFq-nbia-digest-1 (2).xlsx',
 'doiJNLP-QoOaKUdn-nbia-digest-1 (4).xlsx',
 'Integrated_data_2021.RData',
 'table1_baseline.csv',
 '08_make_modeling_table.ipynb',
 'modeling_table.csv',
 'desktop.ini']

In [19]:
import pandas as pd
combined_clinical = pd.read_csv('combined_glioma_who2021.csv')
print(combined_clinical.shape)
combined_clinical.head(5)

(1544, 9)


,case_id,site,glioma_class,idh,codel_1p19q,age,sex,Scanner,field_strength
0,TCGA-06-2570,TCGA,astrocytoma,1,0,21.0,0,GE,-1.0
1,TCGA-06-6389,TCGA,astrocytoma,1,0,49.0,0,GE,-1.0
2,TCGA-14-1456,TCGA,astrocytoma,1,0,23.0,1,Philips,-1.0
3,TCGA-CS-4942,TCGA,astrocytoma,1,0,44.0,0,Philips,-1.0
4,TCGA-CS-4944,TCGA,astrocytoma,1,0,50.0,1,Siemens,-1.0


In [20]:
all_radiomics_features=pd.read_csv('all_radiomics_features.csv')

replacements = {
    'MR_EGD': 'EGD',
    'UCSF-PDGM-0': 'UCSF-PDGM-',
    '_nifti': ''
}
all_radiomics_features['Case_ID'] = all_radiomics_features['Case_ID'].replace(replacements, regex=True)

In [21]:
modeling_table = all_radiomics_features.merge(combined_clinical, how='inner', left_on = 'Case_ID', right_on = 'case_id')
modeling_table.shape

(1544, 4235)

In [22]:
print(modeling_table.shape)
modeling_table.head(5)

(1544, 4235)


,Case_ID,Cohort,FLAIR_original_firstorder_10Percentile,FLAIR_original_firstorder_90Percentile,FLAIR_original_firstorder_Energy,FLAIR_original_firstorder_Entropy,FLAIR_original_firstorder_InterquartileRange,FLAIR_original_firstorder_Kurtosis,FLAIR_original_firstorder_Maximum,FLAIR_original_firstorder_MeanAbsoluteDeviation,...,T2_wavelet-LLL_gldm_SmallDependenceLowGrayLevelEmphasis,case_id,site,glioma_class,idh,codel_1p19q,age,sex,Scanner,field_strength
0,BT0001,UTSW,0.736154,2.818637,571334.478828,5.014156,1.257186,2.394337,4.774386,0.672184,...,0.000328,BT0001,UTSW,glioblastoma,0,0,44.0,0,Siemens,1.5
1,BT0002,UTSW,0.488673,2.230858,23555.120764,4.935280,0.838735,5.300931,3.214054,0.558603,...,0.000553,BT0002,UTSW,glioblastoma,0,0,58.0,1,GE,1.5
2,BT0003,UTSW,0.496658,2.797938,410631.266054,4.877536,1.208744,3.237318,4.928467,0.724986,...,0.000249,BT0003,UTSW,astrocytoma,1,0,22.0,0,GE,3.0
3,BT0005,UTSW,0.464657,2.386642,482694.345238,5.081625,0.996302,3.306076,3.424597,0.599792,...,0.000618,BT0005,UTSW,glioblastoma,0,0,67.0,1,GE,1.5
4,BT0007,UTSW,0.738694,2.959158,751600.309040,4.712456,1.178043,3.096775,4.804285,0.694520,...,0.000460,BT0007,UTSW,astrocytoma,1,0,30.0,1,GE,3.0


In [23]:
modeling_table = modeling_table.drop(['Case_ID', 
                                      'Cohort', 
                                      'idh', 
                                      'codel_1p19q', 
                                      'field_strength'], axis=1)

modeling_table = modeling_table.rename(columns={"glioma_class": "2021_glioma_class"})
modeling_table.head(5)

,FLAIR_original_firstorder_10Percentile,FLAIR_original_firstorder_90Percentile,FLAIR_original_firstorder_Energy,FLAIR_original_firstorder_Entropy,FLAIR_original_firstorder_InterquartileRange,FLAIR_original_firstorder_Kurtosis,FLAIR_original_firstorder_Maximum,FLAIR_original_firstorder_MeanAbsoluteDeviation,FLAIR_original_firstorder_Mean,FLAIR_original_firstorder_Median,...,T2_wavelet-LLL_gldm_LowGrayLevelEmphasis,T2_wavelet-LLL_gldm_SmallDependenceEmphasis,T2_wavelet-LLL_gldm_SmallDependenceHighGrayLevelEmphasis,T2_wavelet-LLL_gldm_SmallDependenceLowGrayLevelEmphasis,case_id,site,2021_glioma_class,age,sex,Scanner
0,0.736154,2.818637,571334.478828,5.014156,1.257186,2.394337,4.774386,0.672184,1.776570,1.808323,...,0.001758,0.161286,151.451947,0.000328,BT0001,UTSW,glioblastoma,44.0,0,Siemens
1,0.488673,2.230858,23555.120764,4.935280,0.838735,5.300931,3.214054,0.558603,1.385858,1.474390,...,0.002055,0.211019,187.833381,0.000553,BT0002,UTSW,glioblastoma,58.0,1,GE
2,0.496658,2.797938,410631.266054,4.877536,1.208744,3.237318,4.928467,0.724986,1.581980,1.541386,...,0.001265,0.130759,147.092505,0.000249,BT0003,UTSW,astrocytoma,22.0,0,GE
3,0.464657,2.386642,482694.345238,5.081625,0.996302,3.306076,3.424597,0.599792,1.374134,1.373485,...,0.003784,0.141160,115.666222,0.000618,BT0005,UTSW,glioblastoma,67.0,1,GE
4,0.738694,2.959158,751600.309040,4.712456,1.178043,3.096775,4.804285,0.694520,1.891049,1.917817,...,0.002084,0.135099,111.399781,0.000460,BT0007,UTSW,astrocytoma,30.0,1,GE


In [24]:
modeling_table.to_csv('modeling_table.csv', index=False)